# Economic Report Q&A Bot (RAG in Python)
**What it does:** You upload a PDF (e.g. RBI Monetary Policy Statement, Economic Survey, a company annual report). Ask a question in plain English and the bot finds the most relevant passages and writes an answer from them.

**Tech:** Python, pypdf, sentence-transformers (embeddings), cosine similarity, Hugging Face flan-t5 (answer generation). No API key needed.

**How to run (works on a phone):** Open in Google Colab, tap *Runtime > Run all*, upload a PDF when asked.

## Step 1: Install libraries

In [1]:
!pip install -q pypdf sentence-transformers transformers

## Step 2: Upload your PDF

In [3]:
from google.colab import files
uploaded = files.upload()
pdf_name = list(uploaded.keys())[0]
print('Loaded:', pdf_name)

Saving Brickwork_Ratings_RBI_MPC_Meet_Outcome_October_2026_copy.pdf to Brickwork_Ratings_RBI_MPC_Meet_Outcome_October_2026_copy.pdf
Loaded: Brickwork_Ratings_RBI_MPC_Meet_Outcome_October_2026_copy.pdf


## Step 3: Read the PDF and split it into chunks
Long documents are split into small overlapping pieces so the bot can find the exact part that answers a question.

In [4]:
from pypdf import PdfReader

reader = PdfReader(pdf_name)
text = ' '.join((page.extract_text() or '') for page in reader.pages)
text = ' '.join(text.split())  # clean extra spaces/newlines

def chunk_text(text, size=120, overlap=30):
    words = text.split()
    chunks, i = [], 0
    while i < len(words):
        chunks.append(' '.join(words[i:i + size]))
        i += size - overlap
    return chunks

chunks = chunk_text(text)
print('Pages:', len(reader.pages), '| Chunks:', len(chunks))

Pages: 12 | Chunks: 51


## Step 4: Turn chunks into embeddings (numbers that capture meaning)

In [5]:
from sentence_transformers import SentenceTransformer
import numpy as np

embedder = SentenceTransformer('all-MiniLM-L6-v2')
chunk_vectors = embedder.encode(chunks, normalize_embeddings=True, show_progress_bar=True)
print('Embedding matrix shape:', chunk_vectors.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Embedding matrix shape: (51, 384)


## Step 5: Retrieve the most relevant chunks for a question

In [6]:
def retrieve(question, k=3):
    q_vec = embedder.encode([question], normalize_embeddings=True)[0]
    scores = chunk_vectors @ q_vec          # cosine similarity
    top = np.argsort(scores)[::-1][:k]
    return [(chunks[i], float(scores[i])) for i in top]

## Step 6: Generate the answer using only the retrieved text

In [9]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

tokenizer = AutoTokenizer.from_pretrained('google/flan-t5-base')
model = AutoModelForSeq2SeqLM.from_pretrained('google/flan-t5-base')

def ask(question, k=3):
    results = retrieve(question, k)
    context = ' '.join(c for c, _ in results)
    prompt = ('Answer the question using only the context. '
              'If the answer is not in the context, say you do not know.\n\n'
              f'Context: {context}\n\nQuestion: {question}\nAnswer:')
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=512)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=150)
    answer = tokenizer.decode(output[0], skip_special_tokens=True)
    print('Q:', question)
    print('A:', answer)
    print('\nSources used:')
    for c, s in results:
        print(f'  (similarity {s:.2f}) {c[:150]}...')

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  990MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

## Step 7: Ask questions!
Change the question below to anything about your PDF.

In [10]:
ask('What did the MPC decide on the repo rate?')

Q: What did the MPC decide on the repo rate?
A: The real GDP of India is estimated to be around $1.3 trillion.

Sources used:
  (similarity 0.53) Inflation is the reason; strong growth gives the RBI room to act. Table 1: Five calls from the October policy For credit, favour strong balance sheets...
  (similarity 0.49) how far prices are spreading 5 04 Liquidity, bonds and the rupee Why the hike is only half transmitted and what drives the rupee 6 05 Sectoral implica...
  (similarity 0.46) points (bps) to 5.50% on 7 October 2026. This is its first hike since February 2023, and the decision was in line with our expectations. The MPC also ...


In [11]:
ask('What is the core inflation projection?')

Q: What is the core inflation projection?
A: Inflation

Sources used:
  (similarity 0.50) and services exports Source: Reserve Bank of India; National Statistics Office; BWR Research Inflation: a supply shock that is starting to spread The ...
  (similarity 0.50) deposits. Media and analysts Track the call rate and core inflation excluding precious metals, not just the repo rate and headline CPI. Source: BWR Re...
  (similarity 0.49) to peak at 6.0% in Q3 FY27. At the same time, the economy is strong. Real GDP grew 7.8% in Q1 FY27, and the RBI has raised its FY27 growth forecast to...


## Ideas to extend (mention these in your README)
- Show page numbers with each source
- Compare answers using different chunk sizes
- Swap flan-t5 for a larger model
- Add a simple Gradio chat interface

## Limitations
Small free models can miss details, so always check the sources shown.